<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC Spot Grid Trading Backtest

**Workflow:** Setup → Excel checkpoint → V0 Fixed Grid → V1 Dynamic Grid → V0/V1 Comparison → System Audit → GitHub Log

- **V0** = fixed arithmetic grid baseline.
- **V1** = dynamic re-centering arithmetic grid.
- V1 keeps order size fixed; **compounding is OFF** so the change being studied is the grid regime.
- Historical 1-minute OHLCV is used for execution simulation.


## 1. Setup & Data

In [ ]:
import os, heapq, bisect, json, base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/03.Trading/00.Live Trading'

SYMBOL = 'BTCUSDT'
START_DATE = '2024-01-01'
END_DATE = '2026-01-01'

BUY_FEE = 0.001
SELL_FEE = 0.001

# KZM Excel checkpoint parameters
GRID_CAPITAL = 3000.0
GRID_CEILING = 8987.0
GRID_FLOOR = 1987.0
GRID_GAP = 70.0

def load_market_data(symbol, timeframe, data_dir):
    path = os.path.join(data_dir, f'{symbol}-{timeframe}-combined.csv')
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    df['open_time'] = pd.to_datetime(df['open_time'], utc=True)
    numeric_cols = ['open', 'high', 'low', 'close', 'volume']
    df[numeric_cols] = df[numeric_cols].astype(float)

    return (
        df.drop_duplicates('open_time')
        .sort_values('open_time')
        .reset_index(drop=True)
    )

df_1m = load_market_data(SYMBOL, '1m', DATA_DIR)

start_ts = pd.Timestamp(START_DATE, tz='UTC')
end_ts = pd.Timestamp(END_DATE, tz='UTC')
df_1m = df_1m.loc[
    (df_1m['open_time'] >= start_ts)
    & (df_1m['open_time'] < end_ts)
].reset_index(drop=True)

print('===== DATA =====')
print(f'Rows: {len(df_1m):,}')
print(f'Period: {df_1m.open_time.min()} -> {df_1m.open_time.max()}')


## 2. Excel Grid Logic

KZM Excel remains the source-of-truth checkpoint for the arithmetic grid formula. The independently known first-grid profit is used to confirm the Python implementation still matches the workbook.


In [ ]:
def build_excel_grid_table(
    capital,
    ceiling,
    floor,
    gap,
    buy_fee=0.001,
    sell_fee=0.001,
):
    if capital <= 0:
        raise ValueError('capital must be greater than 0.')
    if ceiling <= floor:
        raise ValueError('ceiling must be greater than floor.')
    if gap <= 0:
        raise ValueError('gap must be greater than 0.')

    raw_levels = (ceiling - floor) / gap
    if not np.isclose(raw_levels, round(raw_levels)):
        raise ValueError('(ceiling - floor) must be exactly divisible by gap.')

    n_levels = int(round(raw_levels))
    capital_per_level = capital / n_levels

    buy_prices = ceiling - gap * np.arange(1, n_levels + 1)
    sell_prices = buy_prices + gap

    gross_base_amount = capital_per_level / buy_prices
    buy_fee_base = gross_base_amount * buy_fee
    base_amount = gross_base_amount - buy_fee_base

    gross_sell = base_amount * sell_prices
    sell_fee_quote = gross_sell * sell_fee
    net_sell = gross_sell - sell_fee_quote
    profit = net_sell - capital_per_level

    return pd.DataFrame({
        'level': np.arange(1, n_levels + 1),
        'buy_price': buy_prices,
        'sell_price': sell_prices,
        'capital_per_level': capital_per_level,
        'gross_base_amount': gross_base_amount,
        'buy_fee_base': buy_fee_base,
        'base_amount': base_amount,
        'gross_sell': gross_sell,
        'sell_fee_quote': sell_fee_quote,
        'net_sell': net_sell,
        'profit': profit,
    })

df_grid_excel = build_excel_grid_table(
    GRID_CAPITAL, GRID_CEILING, GRID_FLOOR, GRID_GAP, BUY_FEE, SELL_FEE
)

excel_check = df_grid_excel.iloc[0]
EXPECTED_FIRST_PROFIT = 0.1750644398340242

assert np.isclose(excel_check['profit'], EXPECTED_FIRST_PROFIT, atol=1e-12)

print(
    f'Excel checkpoint PASSED: '
    f'{excel_check.buy_price:.0f} -> {excel_check.sell_price:.0f}, '
    f'profit={excel_check.profit:.12f}'
)


## 3. V0 Fixed Grid Configuration

V0 keeps the existing baseline unchanged. Full-period historical Low/High are used only to select the V0 boundaries, so V0 still contains look-ahead bias and is treated as the fixed-grid baseline / execution reference.


In [ ]:
BACKTEST_CAPITAL = 3000.0
BACKTEST_GAP = 1000.0
PRICE_ROUNDING = 1000.0

historical_low = df_1m['low'].min()
historical_high = df_1m['high'].max()

BACKTEST_FLOOR = (
    np.floor(historical_low / PRICE_ROUNDING) * PRICE_ROUNDING
)
BACKTEST_CEILING = (
    np.ceil(historical_high / PRICE_ROUNDING) * PRICE_ROUNDING
)

raw_backtest_levels = (
    BACKTEST_CEILING - BACKTEST_FLOOR
) / BACKTEST_GAP

if not np.isclose(raw_backtest_levels, round(raw_backtest_levels)):
    raise ValueError('Backtest range must be divisible by BACKTEST_GAP.')

NUMBER_OF_GRIDS = int(round(raw_backtest_levels))
CAPITAL_PER_LEVEL = BACKTEST_CAPITAL / NUMBER_OF_GRIDS

df_grid_backtest = build_excel_grid_table(
    BACKTEST_CAPITAL,
    BACKTEST_CEILING,
    BACKTEST_FLOOR,
    BACKTEST_GAP,
    BUY_FEE,
    SELL_FEE,
)

print('===== V0 FIXED GRID CONFIGURATION =====')
print(f'Historical Low     : {historical_low:,.2f} USDT')
print(f'Historical High    : {historical_high:,.2f} USDT')
print(f'Grid Floor         : {BACKTEST_FLOOR:,.2f} USDT')
print(f'Grid Ceiling       : {BACKTEST_CEILING:,.2f} USDT')
print(f'Grid Gap           : {BACKTEST_GAP:,.2f} USDT')
print(f'Number of Grids    : {NUMBER_OF_GRIDS}')
print(f'Capital            : {BACKTEST_CAPITAL:,.2f} USDT')
print(f'Capital / Grid     : {CAPITAL_PER_LEVEL:,.6f} USDT')


## 4. V0 Fixed Grid Engine — 1 Minute

Execution rules:
- BUY only on downward crossing of a grid level.
- Existing SELL targets can fill when candle High reaches the target.
- A newly opened BUY cannot SELL in the same candle.
- A sold grid cannot rebuy in the same candle.
- Same-candle SELL proceeds are not reused for BUYs.
- BUY fee is deducted from BTC; SELL fee is deducted from USDT.


In [ ]:
def run_grid_backtest(df_price, grid_table, initial_capital):
    required = {'open_time', 'open', 'high', 'low', 'close'}
    missing = required.difference(df_price.columns)

    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    if len(df_price) == 0:
        raise ValueError('df_price is empty.')
    if initial_capital <= 0:
        raise ValueError('initial_capital must be greater than 0.')

    data = df_price.sort_values('open_time').reset_index(drop=True)
    grid = grid_table.sort_values('buy_price').reset_index(drop=True).copy()

    if len(grid) == 0:
        raise ValueError('grid_table is empty.')

    buy_prices = grid['buy_price'].to_numpy(float)
    sell_prices = grid['sell_price'].to_numpy(float)
    capital_per_level = grid['capital_per_level'].to_numpy(float)
    base_amount = grid['base_amount'].to_numpy(float)
    buy_fee_base = grid['buy_fee_base'].to_numpy(float)
    sell_fee_quote = grid['sell_fee_quote'].to_numpy(float)
    net_sell = grid['net_sell'].to_numpy(float)
    cycle_profit = grid['profit'].to_numpy(float)
    excel_level = grid['level'].to_numpy(int)

    if (
        len(grid) > 1
        and not np.allclose(np.diff(buy_prices), np.diff(buy_prices)[0])
    ):
        raise ValueError('Arithmetic grid required.')

    holding = np.zeros(len(grid), dtype=bool)
    buy_time = [None] * len(grid)
    sell_heap = []

    cash = float(initial_capital)
    open_btc = 0.0
    realized_profit = 0.0
    total_buy_fee_btc = 0.0
    total_buy_fee_usdt_equiv = 0.0
    total_sell_fee_usdt = 0.0
    completed_cycles = 0

    trade_events = []
    completed_trades = []

    n_rows = len(data)
    equity_values = np.empty(n_rows)
    cash_values = np.empty(n_rows)
    btc_values = np.empty(n_rows)

    buy_price_list = buy_prices.tolist()
    prev_close = None
    event_id = 0

    for i, row in enumerate(data.itertuples(index=False)):
        timestamp = row.open_time
        open_price = float(row.open)
        high_price = float(row.high)
        low_price = float(row.low)
        close_price = float(row.close)

        cash_at_candle_start = cash
        sold_this_candle = set()

        # 1) Existing SELL orders.
        while sell_heap and sell_heap[0][0] <= high_price:
            _, k = heapq.heappop(sell_heap)

            if not holding[k]:
                continue

            cash_before = cash
            btc_before = open_btc

            holding[k] = False
            cash += net_sell[k]
            open_btc -= base_amount[k]

            if abs(open_btc) < 1e-12:
                open_btc = 0.0

            realized_profit += cycle_profit[k]
            total_sell_fee_usdt += sell_fee_quote[k]
            completed_cycles += 1
            sold_this_candle.add(k)
            event_id += 1

            completed_trades.append({
                'grid_level': int(excel_level[k]),
                'buy_time': buy_time[k],
                'sell_time': timestamp,
                'buy_price': buy_prices[k],
                'sell_price': sell_prices[k],
                'cost': capital_per_level[k],
                'quote_cost': capital_per_level[k],
                'base_amount': base_amount[k],
                'actual_earn': net_sell[k],
                'net_sell': net_sell[k],
                'grid_cashflow': cycle_profit[k],
                'profit': cycle_profit[k],
            })

            trade_events.append({
                'event_id': event_id,
                'time': timestamp,
                'side': 'SELL',
                'grid_level': int(excel_level[k]),
                'price': sell_prices[k],
                'base_amount': base_amount[k],
                'quote_amount': net_sell[k],
                'fee_base': 0.0,
                'fee_quote': sell_fee_quote[k],
                'realized_profit': cycle_profit[k],
                'cash_movement': net_sell[k],
                'grid_cashflow': cycle_profit[k],
                'cash_before': cash_before,
                'cash_after': cash,
                'btc_before': btc_before,
                'btc_after': open_btc,
            })

            buy_time[k] = None

        # 2) Downward BUY crossings.
        # Same-candle SELL proceeds are intentionally unavailable.
        buy_budget = cash_at_candle_start
        down_start = (
            open_price if prev_close is None else max(prev_close, open_price)
        )

        if low_price < down_start:
            first_idx = bisect.bisect_left(buy_price_list, low_price)
            stop_idx = bisect.bisect_left(buy_price_list, down_start)

            # Higher levels are crossed first on the way down.
            for k in range(stop_idx - 1, first_idx - 1, -1):
                if holding[k] or k in sold_this_candle:
                    continue

                cost = capital_per_level[k]

                if buy_budget + 1e-12 < cost:
                    break

                cash_before = cash
                btc_before = open_btc

                holding[k] = True
                buy_time[k] = timestamp

                buy_budget -= cost
                cash -= cost
                open_btc += base_amount[k]

                total_buy_fee_btc += buy_fee_base[k]
                total_buy_fee_usdt_equiv += buy_fee_base[k] * buy_prices[k]

                heapq.heappush(sell_heap, (sell_prices[k], k))
                event_id += 1

                trade_events.append({
                    'event_id': event_id,
                    'time': timestamp,
                    'side': 'BUY',
                    'grid_level': int(excel_level[k]),
                    'price': buy_prices[k],
                    'base_amount': base_amount[k],
                    'quote_amount': cost,
                    'fee_base': buy_fee_base[k],
                    'fee_quote': 0.0,
                    'realized_profit': 0.0,
                    'cash_movement': -cost,
                    'grid_cashflow': 0.0,
                    'cash_before': cash_before,
                    'cash_after': cash,
                    'btc_before': btc_before,
                    'btc_after': open_btc,
                })

        # 3) Mark portfolio to market at candle Close.
        equity_values[i] = cash + open_btc * close_price
        cash_values[i] = cash
        btc_values[i] = open_btc
        prev_close = close_price

    equity_curve = pd.DataFrame({
        'open_time': data['open_time'].to_numpy(),
        'close': data['close'].to_numpy(float),
        'cash': cash_values,
        'btc': btc_values,
        'equity': equity_values,
    })

    running_peak = np.maximum.accumulate(equity_values)
    drawdown = equity_values / running_peak - 1.0
    equity_curve['drawdown'] = drawdown

    max_drawdown = float(drawdown.min())
    final_equity = float(equity_values[-1])
    net_return = final_equity / initial_capital - 1.0

    elapsed_days = (
        data['open_time'].iloc[-1] - data['open_time'].iloc[0]
    ).total_seconds() / 86400.0

    annualized_return = np.nan
    if elapsed_days > 0 and final_equity > 0:
        annualized_log_growth = (
            np.log(final_equity / initial_capital) * (365.25 / elapsed_days)
        )
        if annualized_log_growth < 700:
            annualized_return = float(np.expm1(annualized_log_growth))

    calmar_ratio = np.nan
    if max_drawdown < 0 and np.isfinite(annualized_return):
        calmar_ratio = float(annualized_return / abs(max_drawdown))

    trade_log = pd.DataFrame(trade_events)
    if not trade_log.empty:
        trade_log['cumulative_cash_movement'] = (
            trade_log['cash_movement'].cumsum()
        )
        trade_log['cumulative_grid_cashflow'] = (
            trade_log['grid_cashflow'].cumsum()
        )

    summary = {
        'initial_capital': float(initial_capital),
        'final_equity': final_equity,
        'net_return': float(net_return),
        'annualized_return': annualized_return,
        'max_drawdown': max_drawdown,
        'calmar_ratio': calmar_ratio,
        'completed_cycles': int(completed_cycles),
        'open_positions': int(holding.sum()),
        'final_cash': float(cash),
        'final_btc': float(open_btc),
        'realized_profit': float(realized_profit),
        'unrealized_pnl': float(
            final_equity - initial_capital - realized_profit
        ),
        'buy_fee_btc': float(total_buy_fee_btc),
        'buy_fee_usdt_equiv': float(total_buy_fee_usdt_equiv),
        'sell_fee_usdt': float(total_sell_fee_usdt),
        'total_fee_usdt_equiv': float(
            total_buy_fee_usdt_equiv + total_sell_fee_usdt
        ),
    }

    return {
        'summary': summary,
        'trade_log': trade_log,
        'completed_trades': pd.DataFrame(completed_trades),
        'equity_curve': equity_curve,
        'grid_state': grid.assign(holding=holding, buy_time=buy_time),
    }


## 5. Verification Test Report — Human-readable

Small deterministic price paths with known answers are used to verify the core V0 execution logic independently of the two-year historical result.


In [ ]:
def make_test_candles(rows, start='2024-01-01'):
    rows = list(rows)
    return pd.DataFrame({
        'open_time': pd.date_range(start, periods=len(rows), freq='min', tz='UTC'),
        'open': [r[0] for r in rows],
        'high': [r[1] for r in rows],
        'low': [r[2] for r in rows],
        'close': [r[3] for r in rows],
    })

test_grid = build_excel_grid_table(
    capital=300.0,
    ceiling=130.0,
    floor=100.0,
    gap=10.0,
    buy_fee=0.001,
    sell_fee=0.001,
)

up_only = run_grid_backtest(
    make_test_candles([(115, 125, 115, 122)]),
    test_grid,
    300.0,
)

down_one = run_grid_backtest(
    make_test_candles([(125, 126, 115, 118)]),
    test_grid,
    300.0,
)

same_candle = run_grid_backtest(
    make_test_candles([(125, 135, 115, 120)]),
    test_grid,
    300.0,
)

later_sell = run_grid_backtest(
    make_test_candles([
        (125, 126, 115, 118),
        (120, 131, 120, 130),
    ]),
    test_grid,
    300.0,
)

verification_rows = [
    {
        'Scenario': 'Excel first-grid profit',
        'Expected': '0.175064439834',
        'Actual': f'{excel_check.profit:.12f}',
        'Result': 'PASS' if np.isclose(
            excel_check.profit, EXPECTED_FIRST_PROFIT, atol=1e-12
        ) else 'FAIL',
    },
    {
        'Scenario': 'Upward-only candle',
        'Expected': 'BUY count = 0',
        'Actual': f"BUY count = {int(up_only['trade_log'].shape[0])}",
        'Result': 'PASS' if up_only['trade_log'].empty else 'FAIL',
    },
    {
        'Scenario': 'Downward 125 -> 115 crossing',
        'Expected': 'BUY 120 only',
        'Actual': (
            f"BUY {down_one['trade_log'].iloc[0]['price']:.0f}"
            if len(down_one['trade_log']) else 'no BUY'
        ),
        'Result': (
            'PASS'
            if len(down_one['trade_log']) == 1
            and np.isclose(down_one['trade_log'].iloc[0]['price'], 120.0)
            else 'FAIL'
        ),
    },
    {
        'Scenario': 'Same-candle BUY -> SELL protection',
        'Expected': 'SELL count = 0',
        'Actual': (
            f"SELL count = {int(same_candle['trade_log']['side'].eq('SELL').sum())}"
        ),
        'Result': (
            'PASS'
            if int(same_candle['trade_log']['side'].eq('SELL').sum()) == 0
            else 'FAIL'
        ),
    },
    {
        'Scenario': 'Later-candle completed cycle',
        'Expected': 'BUY 120 -> SELL 130',
        'Actual': ' -> '.join(
            f"{row.side} {row.price:.0f}"
            for row in later_sell['trade_log'].itertuples()
        ),
        'Result': (
            'PASS'
            if later_sell['trade_log']['side'].tolist() == ['BUY', 'SELL']
            and np.isclose(
                later_sell['summary']['realized_profit'],
                8.116775,
                atol=1e-12,
            )
            else 'FAIL'
        ),
    },
]

df_verification_report = pd.DataFrame(verification_rows)
display(df_verification_report)

verification_failed = df_verification_report.loc[
    df_verification_report['Result'].ne('PASS')
]
if len(verification_failed):
    raise AssertionError('V0 VERIFICATION REPORT FAILED')

print('V0 VERIFICATION: PASS')


## 6. Run V0 Fixed Grid

In [ ]:
v0 = run_grid_backtest(
    df_1m,
    df_grid_backtest,
    BACKTEST_CAPITAL,
)
v0_summary = v0['summary']

print('===== V0 BACKTEST SUMMARY =====')
print(f"Initial Capital     : {v0_summary['initial_capital']:,.2f} USDT")
print(f"Final Equity        : {v0_summary['final_equity']:,.2f} USDT")
print(f"Net Return          : {v0_summary['net_return']:.2%}")
print(f"Annualized Return   : {v0_summary['annualized_return']:.2%}")
print(f"Max Drawdown        : {v0_summary['max_drawdown']:.2%}")
print(f"Calmar Ratio        : {v0_summary['calmar_ratio']:.3f}")
print(f"Completed Cycles    : {v0_summary['completed_cycles']:,}")
print(f"Open Positions      : {v0_summary['open_positions']:,}")
print(f"Final Cash          : {v0_summary['final_cash']:,.2f} USDT")
print(f"Final BTC           : {v0_summary['final_btc']:.8f} BTC")
print(f"Realized Profit     : {v0_summary['realized_profit']:,.2f} USDT")
print(f"Unrealized P&L      : {v0_summary['unrealized_pnl']:,.2f} USDT")
print(f"Total Fee           : {v0_summary['total_fee_usdt_equiv']:,.2f} USDT")


## 7. V0 Cashflow & Monthly Portfolio P&L

In [ ]:
def build_monthly_portfolio_pnl(
    equity_curve,
    initial_capital,
    start_date=None,
    end_date=None,
):
    eq = equity_curve[['open_time', 'equity']].copy()
    eq['open_time'] = pd.to_datetime(eq['open_time'], utc=True)

    if start_date is not None:
        eq = eq.loc[
            eq['open_time'] >= pd.Timestamp(start_date, tz='UTC')
        ]
    if end_date is not None:
        eq = eq.loc[
            eq['open_time'] < pd.Timestamp(end_date, tz='UTC')
        ]

    eq['month'] = eq['open_time'].dt.strftime('%Y-%m')
    monthly = (
        eq.groupby('month', as_index=False)['equity']
        .last()
        .rename(columns={'equity': 'ending_equity'})
    )

    monthly['beginning_equity'] = monthly['ending_equity'].shift(1)
    if len(monthly):
        monthly.loc[monthly.index[0], 'beginning_equity'] = initial_capital

    monthly['portfolio_net_pnl'] = (
        monthly['ending_equity'] - monthly['beginning_equity']
    )
    monthly['monthly_return'] = (
        monthly['portfolio_net_pnl'] / monthly['beginning_equity']
    )
    monthly['cumulative_portfolio_pnl'] = (
        monthly['portfolio_net_pnl'].cumsum()
    )

    return monthly[
        [
            'month',
            'beginning_equity',
            'ending_equity',
            'portfolio_net_pnl',
            'monthly_return',
            'cumulative_portfolio_pnl',
        ]
    ]

df_v0_trade_log = v0['trade_log'].copy()
df_v0_completed = v0['completed_trades'].copy()
df_v0_grid_state = v0['grid_state'].copy()

if len(df_v0_completed):
    df_v0_completed['month'] = (
        pd.to_datetime(df_v0_completed['sell_time'], utc=True)
        .dt.strftime('%Y-%m')
    )
    df_v0_grid_cashflow_monthly = (
        df_v0_completed.groupby('month', as_index=False)
        .agg(
            monthly_grid_cashflow=('grid_cashflow', 'sum'),
            completed_cycles=('grid_cashflow', 'size'),
        )
    )
    df_v0_grid_cashflow_monthly['cumulative_grid_cashflow'] = (
        df_v0_grid_cashflow_monthly['monthly_grid_cashflow'].cumsum()
    )
else:
    df_v0_grid_cashflow_monthly = pd.DataFrame(
        columns=[
            'month',
            'monthly_grid_cashflow',
            'completed_cycles',
            'cumulative_grid_cashflow',
        ]
    )

df_v0_monthly = build_monthly_portfolio_pnl(
    v0['equity_curve'],
    BACKTEST_CAPITAL,
    START_DATE,
    END_DATE,
)

print('===== V0 MONTHLY GRID CASHFLOW =====')
display(df_v0_grid_cashflow_monthly.round(6))

print('===== V0 MONTHLY PORTFOLIO P&L =====')
display(df_v0_monthly.round(6))


## 8. V1 Dynamic Grid Configuration

V1 changes the active grid regime causally without using future prices.

Current V1 settings:
- Gap = 1,000 USDT
- 30 active BUY→SELL intervals
- Re-center when candle Close moves 5 grid gaps from the current reference
- Re-center becomes effective on the next candle
- Existing positions keep their original SELL targets
- Fixed order size = Initial Capital / 30
- Compounding = OFF


In [ ]:
V1_GRID_GAP = BACKTEST_GAP
V1_NUMBER_OF_GRIDS = 30
V1_RECENTER_TRIGGER_GRIDS = 5
V1_CAPITAL_PER_GRID = BACKTEST_CAPITAL / V1_NUMBER_OF_GRIDS

print('===== V1 DYNAMIC GRID CONFIGURATION =====')
print(f'Gap                    : {V1_GRID_GAP:,.0f} USDT')
print(f'Active Grid Intervals  : {V1_NUMBER_OF_GRIDS}')
print(f'Capital / Grid         : {V1_CAPITAL_PER_GRID:,.6f} USDT')
print(
    f'Recenter Trigger       : {V1_RECENTER_TRIGGER_GRIDS} grids '
    f'({V1_RECENTER_TRIGGER_GRIDS * V1_GRID_GAP:,.0f} USDT)'
)
print('Compounding             : OFF')


## 9. V1 Dynamic Re-centering Grid Engine

Dynamic rule:
- Initial reference uses the first candle Open only.
- Re-center decision uses the current candle Close only.
- New regime is effective from the next candle.
- Old positions are not reset and retain their original SELL targets.


In [ ]:
def round_to_gap(price, gap):
    if gap <= 0:
        raise ValueError('gap must be greater than 0.')
    return float(np.floor(float(price) / gap + 0.5) * gap)


def build_dynamic_regime(
    reference_price,
    gap,
    number_of_grids,
    capital,
    regime_id=0,
):
    if capital <= 0:
        raise ValueError('capital must be greater than 0.')
    if gap <= 0:
        raise ValueError('gap must be greater than 0.')
    if number_of_grids < 2:
        raise ValueError('number_of_grids must be at least 2.')

    lower_grids = number_of_grids // 2
    upper_grids = number_of_grids - lower_grids

    floor = float(reference_price - lower_grids * gap)
    ceiling = float(reference_price + upper_grids * gap)

    if floor <= 0:
        raise ValueError(
            'Dynamic grid floor must stay above 0. '
            'Reduce number_of_grids or gap.'
        )

    buy_prices = np.arange(floor, ceiling, gap, dtype=float)
    if len(buy_prices) != number_of_grids:
        raise AssertionError('Dynamic regime grid count mismatch.')

    return {
        'regime_id': int(regime_id),
        'reference_price': float(reference_price),
        'floor': floor,
        'ceiling': ceiling,
        'buy_prices': buy_prices,
        'capital_per_grid': float(capital / number_of_grids),
        'gap': float(gap),
    }


def run_dynamic_grid_backtest(
    df_price,
    initial_capital,
    gap,
    number_of_grids,
    recenter_trigger_grids,
    buy_fee=0.001,
    sell_fee=0.001,
):
    required = {'open_time', 'open', 'high', 'low', 'close'}
    missing = required.difference(df_price.columns)

    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    if len(df_price) == 0:
        raise ValueError('df_price is empty.')
    if initial_capital <= 0:
        raise ValueError('initial_capital must be greater than 0.')
    if gap <= 0:
        raise ValueError('gap must be greater than 0.')
    if number_of_grids < 2:
        raise ValueError('number_of_grids must be at least 2.')
    if recenter_trigger_grids < 1:
        raise ValueError('recenter_trigger_grids must be at least 1.')
    if not (0 <= buy_fee < 1 and 0 <= sell_fee < 1):
        raise ValueError('fees must be in [0, 1).')

    data = df_price.sort_values('open_time').reset_index(drop=True)

    initial_reference = round_to_gap(float(data.iloc[0]['open']), gap)
    regime_id = 0
    regime = build_dynamic_regime(
        initial_reference,
        gap,
        number_of_grids,
        initial_capital,
        regime_id,
    )

    regime_history = [{
        'regime_id': 0,
        'effective_time': data.iloc[0]['open_time'],
        'reference_price': regime['reference_price'],
        'floor': regime['floor'],
        'ceiling': regime['ceiling'],
        'reason': 'INITIAL',
    }]
    recenter_events = []

    cash = float(initial_capital)
    open_btc = 0.0
    realized_profit = 0.0
    total_buy_fee_btc = 0.0
    total_buy_fee_usdt_equiv = 0.0
    total_sell_fee_usdt = 0.0
    completed_cycles = 0

    event_id = 0
    position_id = 0
    positions = {}
    open_by_buy_price = {}
    sell_heap = []
    trade_events = []
    completed_trades = []

    n_rows = len(data)
    equity_values = np.empty(n_rows)
    cash_values = np.empty(n_rows)
    btc_values = np.empty(n_rows)
    reference_values = np.empty(n_rows)
    regime_values = np.empty(n_rows, dtype=int)

    prev_close = None
    tolerance = 1e-9

    for i, row in enumerate(data.itertuples(index=False)):
        timestamp = row.open_time
        open_price = float(row.open)
        high_price = float(row.high)
        low_price = float(row.low)
        close_price = float(row.close)

        cash_at_candle_start = cash
        sold_this_candle = set()

        # 1) Existing SELL orders, including positions from old regimes.
        while sell_heap and sell_heap[0][0] <= high_price + tolerance:
            _, pid = heapq.heappop(sell_heap)
            pos = positions.get(pid)

            if pos is None or not pos['is_open']:
                continue

            cash_before = cash
            btc_before = open_btc

            pos['is_open'] = False
            cash += pos['net_sell']
            open_btc -= pos['base_amount']
            if abs(open_btc) < 1e-12:
                open_btc = 0.0

            realized_profit += pos['profit']
            total_sell_fee_usdt += pos['sell_fee_quote']
            completed_cycles += 1
            sold_this_candle.add(pos['buy_price'])
            open_by_buy_price.pop(pos['buy_price'], None)
            event_id += 1

            completed_trades.append({
                'position_id': pid,
                'regime_id': pos['regime_id'],
                'buy_time': pos['buy_time'],
                'sell_time': timestamp,
                'buy_price': pos['buy_price'],
                'sell_price': pos['sell_price'],
                'cost': pos['cost'],
                'quote_cost': pos['cost'],
                'base_amount': pos['base_amount'],
                'actual_earn': pos['net_sell'],
                'net_sell': pos['net_sell'],
                'grid_cashflow': pos['profit'],
                'profit': pos['profit'],
            })

            trade_events.append({
                'event_id': event_id,
                'time': timestamp,
                'side': 'SELL',
                'position_id': pid,
                'regime_id': pos['regime_id'],
                'reference_price': pos['reference_price'],
                'price': pos['sell_price'],
                'base_amount': pos['base_amount'],
                'quote_amount': pos['net_sell'],
                'fee_base': 0.0,
                'fee_quote': pos['sell_fee_quote'],
                'realized_profit': pos['profit'],
                'cash_movement': pos['net_sell'],
                'grid_cashflow': pos['profit'],
                'cash_before': cash_before,
                'cash_after': cash,
                'btc_before': btc_before,
                'btc_after': open_btc,
            })

        # 2) Downward BUY crossings in the current regime.
        buy_budget = cash_at_candle_start
        down_start = (
            open_price if prev_close is None else max(prev_close, open_price)
        )

        buy_prices = regime['buy_prices']
        buy_price_list = buy_prices.tolist()

        if low_price < down_start:
            first_idx = bisect.bisect_left(buy_price_list, low_price)
            stop_idx = bisect.bisect_left(buy_price_list, down_start)

            for k in range(stop_idx - 1, first_idx - 1, -1):
                buy_price = float(buy_prices[k])

                if (
                    buy_price in open_by_buy_price
                    or buy_price in sold_this_candle
                ):
                    continue

                cost = regime['capital_per_grid']
                if buy_budget + 1e-12 < cost:
                    break

                sell_price = buy_price + gap
                gross_base_amount = cost / buy_price
                buy_fee_base = gross_base_amount * buy_fee
                base_amount = gross_base_amount - buy_fee_base

                gross_sell = base_amount * sell_price
                sell_fee_quote = gross_sell * sell_fee
                net_sell = gross_sell - sell_fee_quote
                cycle_profit = net_sell - cost

                cash_before = cash
                btc_before = open_btc

                buy_budget -= cost
                cash -= cost
                open_btc += base_amount
                total_buy_fee_btc += buy_fee_base
                total_buy_fee_usdt_equiv += buy_fee_base * buy_price

                position_id += 1
                pos = {
                    'position_id': position_id,
                    'regime_id': regime['regime_id'],
                    'reference_price': regime['reference_price'],
                    'buy_time': timestamp,
                    'buy_price': buy_price,
                    'sell_price': sell_price,
                    'cost': cost,
                    'base_amount': base_amount,
                    'buy_fee_base': buy_fee_base,
                    'sell_fee_quote': sell_fee_quote,
                    'net_sell': net_sell,
                    'profit': cycle_profit,
                    'is_open': True,
                }
                positions[position_id] = pos
                open_by_buy_price[buy_price] = position_id
                heapq.heappush(sell_heap, (sell_price, position_id))
                event_id += 1

                trade_events.append({
                    'event_id': event_id,
                    'time': timestamp,
                    'side': 'BUY',
                    'position_id': position_id,
                    'regime_id': regime['regime_id'],
                    'reference_price': regime['reference_price'],
                    'price': buy_price,
                    'base_amount': base_amount,
                    'quote_amount': cost,
                    'fee_base': buy_fee_base,
                    'fee_quote': 0.0,
                    'realized_profit': 0.0,
                    'cash_movement': -cost,
                    'grid_cashflow': 0.0,
                    'cash_before': cash_before,
                    'cash_after': cash,
                    'btc_before': btc_before,
                    'btc_after': open_btc,
                })

        # 3) Mark portfolio to market at Close.
        equity_values[i] = cash + open_btc * close_price
        cash_values[i] = cash
        btc_values[i] = open_btc
        reference_values[i] = regime['reference_price']
        regime_values[i] = regime['regime_id']

        # 4) Causal re-center. Decision now, effective next candle.
        trigger_distance = recenter_trigger_grids * gap
        if (
            close_price >= regime['reference_price'] + trigger_distance
            or close_price <= regime['reference_price'] - trigger_distance
        ):
            new_reference = round_to_gap(close_price, gap)

            if new_reference != regime['reference_price']:
                old_regime = regime
                regime_id += 1
                regime = build_dynamic_regime(
                    new_reference,
                    gap,
                    number_of_grids,
                    initial_capital,
                    regime_id,
                )
                direction = (
                    'UP'
                    if new_reference > old_regime['reference_price']
                    else 'DOWN'
                )

                recenter_events.append({
                    'decision_time': timestamp,
                    'direction': direction,
                    'close': close_price,
                    'old_regime_id': old_regime['regime_id'],
                    'new_regime_id': regime_id,
                    'old_reference': old_regime['reference_price'],
                    'new_reference': new_reference,
                    'old_floor': old_regime['floor'],
                    'old_ceiling': old_regime['ceiling'],
                    'new_floor': regime['floor'],
                    'new_ceiling': regime['ceiling'],
                })
                regime_history.append({
                    'regime_id': regime_id,
                    'effective_time': (
                        data.iloc[i + 1]['open_time']
                        if i + 1 < n_rows
                        else pd.NaT
                    ),
                    'reference_price': regime['reference_price'],
                    'floor': regime['floor'],
                    'ceiling': regime['ceiling'],
                    'reason': f'RECENTER_{direction}',
                })

        prev_close = close_price

    equity_curve = pd.DataFrame({
        'open_time': data['open_time'].to_numpy(),
        'close': data['close'].to_numpy(float),
        'cash': cash_values,
        'btc': btc_values,
        'equity': equity_values,
        'reference_price': reference_values,
        'regime_id': regime_values,
    })

    running_peak = np.maximum.accumulate(equity_values)
    drawdown = equity_values / running_peak - 1.0
    equity_curve['drawdown'] = drawdown

    max_drawdown = float(drawdown.min())
    final_equity = float(equity_values[-1])
    net_return = final_equity / initial_capital - 1.0

    elapsed_days = (
        data['open_time'].iloc[-1] - data['open_time'].iloc[0]
    ).total_seconds() / 86400.0

    annualized_return = np.nan
    if elapsed_days > 0 and final_equity > 0:
        annualized_log_growth = (
            np.log(final_equity / initial_capital) * (365.25 / elapsed_days)
        )
        if annualized_log_growth < 700:
            annualized_return = float(np.expm1(annualized_log_growth))

    calmar_ratio = np.nan
    if max_drawdown < 0 and np.isfinite(annualized_return):
        calmar_ratio = float(annualized_return / abs(max_drawdown))

    trade_log = pd.DataFrame(trade_events)
    if not trade_log.empty:
        trade_log['cumulative_cash_movement'] = (
            trade_log['cash_movement'].cumsum()
        )
        trade_log['cumulative_grid_cashflow'] = (
            trade_log['grid_cashflow'].cumsum()
        )

    open_positions = [p for p in positions.values() if p['is_open']]

    summary = {
        'initial_capital': float(initial_capital),
        'final_equity': final_equity,
        'net_return': float(net_return),
        'annualized_return': annualized_return,
        'max_drawdown': max_drawdown,
        'calmar_ratio': calmar_ratio,
        'completed_cycles': int(completed_cycles),
        'open_positions': int(len(open_positions)),
        'final_cash': float(cash),
        'final_btc': float(open_btc),
        'realized_profit': float(realized_profit),
        'unrealized_pnl': float(
            final_equity - initial_capital - realized_profit
        ),
        'buy_fee_btc': float(total_buy_fee_btc),
        'buy_fee_usdt_equiv': float(total_buy_fee_usdt_equiv),
        'sell_fee_usdt': float(total_sell_fee_usdt),
        'total_fee_usdt_equiv': float(
            total_buy_fee_usdt_equiv + total_sell_fee_usdt
        ),
        'recenter_count': int(len(recenter_events)),
    }

    return {
        'summary': summary,
        'trade_log': trade_log,
        'completed_trades': pd.DataFrame(completed_trades),
        'equity_curve': equity_curve,
        'open_positions': pd.DataFrame(open_positions),
        'recenter_log': pd.DataFrame(recenter_events),
        'regime_history': pd.DataFrame(regime_history),
    }


## 10. V1 Verification Test Report — Human-readable

This verifies the new dynamic behavior before the historical V1 run: causal re-centering, preservation of old positions, and preservation of their original SELL target.


In [ ]:
synthetic_v1 = pd.DataFrame({
    'open_time': pd.date_range(
        '2024-01-01', periods=2, freq='min', tz='UTC'
    ),
    'open': [100.0, 90.0],
    'high': [100.0, 100.0],
    'low': [89.0, 89.0],
    'close': [90.0, 90.0],
})

verify_v1 = run_dynamic_grid_backtest(
    synthetic_v1,
    initial_capital=400.0,
    gap=10.0,
    number_of_grids=4,
    recenter_trigger_grids=1,
    buy_fee=0.001,
    sell_fee=0.001,
)

verify_completed = verify_v1['completed_trades']
verify_recenter = verify_v1['recenter_log']

v1_verification_rows = [
    {
        'Scenario': 'Recenter occurred',
        'Expected': '>= 1',
        'Actual': str(len(verify_recenter)),
        'Result': 'PASS' if len(verify_recenter) >= 1 else 'FAIL',
    },
    {
        'Scenario': 'Old regime position preserved',
        'Expected': 'regime_id = 0',
        'Actual': (
            f"regime_id = {int(verify_completed.iloc[0]['regime_id'])}"
            if len(verify_completed) else 'no completed trade'
        ),
        'Result': (
            'PASS'
            if len(verify_completed)
            and int(verify_completed.iloc[0]['regime_id']) == 0
            else 'FAIL'
        ),
    },
    {
        'Scenario': 'Original SELL target preserved',
        'Expected': '90 -> 100',
        'Actual': (
            f"{verify_completed.iloc[0]['buy_price']:.0f} -> "
            f"{verify_completed.iloc[0]['sell_price']:.0f}"
            if len(verify_completed) else 'no completed trade'
        ),
        'Result': (
            'PASS'
            if len(verify_completed)
            and np.isclose(verify_completed.iloc[0]['buy_price'], 90.0)
            and np.isclose(verify_completed.iloc[0]['sell_price'], 100.0)
            else 'FAIL'
        ),
    },
]

df_v1_verification = pd.DataFrame(v1_verification_rows)
display(df_v1_verification)

v1_verification_failed = df_v1_verification.loc[
    df_v1_verification['Result'].ne('PASS')
]
if len(v1_verification_failed):
    raise AssertionError('V1 VERIFICATION REPORT FAILED')

print('V1 VERIFICATION: PASS')


## 11. Run V1 Dynamic Grid

In [ ]:
v1 = run_dynamic_grid_backtest(
    df_1m,
    initial_capital=BACKTEST_CAPITAL,
    gap=V1_GRID_GAP,
    number_of_grids=V1_NUMBER_OF_GRIDS,
    recenter_trigger_grids=V1_RECENTER_TRIGGER_GRIDS,
    buy_fee=BUY_FEE,
    sell_fee=SELL_FEE,
)
v1_summary = v1['summary']

print('===== V1 DYNAMIC GRID SUMMARY =====')
print(f"Initial Capital     : {v1_summary['initial_capital']:,.2f} USDT")
print(f"Final Equity        : {v1_summary['final_equity']:,.2f} USDT")
print(f"Net Return          : {v1_summary['net_return']:.2%}")
print(f"Annualized Return   : {v1_summary['annualized_return']:.2%}")
print(f"Max Drawdown        : {v1_summary['max_drawdown']:.2%}")
print(f"Calmar Ratio        : {v1_summary['calmar_ratio']:.3f}")
print(f"Completed Cycles    : {v1_summary['completed_cycles']:,}")
print(f"Open Positions      : {v1_summary['open_positions']:,}")
print(f"Recenter Count      : {v1_summary['recenter_count']:,}")
print(f"Final Cash          : {v1_summary['final_cash']:,.2f} USDT")
print(f"Final BTC           : {v1_summary['final_btc']:.8f} BTC")
print(f"Realized Profit     : {v1_summary['realized_profit']:,.2f} USDT")
print(f"Unrealized P&L      : {v1_summary['unrealized_pnl']:,.2f} USDT")
print(f"Total Fee           : {v1_summary['total_fee_usdt_equiv']:,.2f} USDT")


## 12. V0 vs V1 Comparison

Both strategies use the same historical data, fees, initial capital, and 1-minute execution semantics. V1 changes the grid regime and also uses 30 active intervals instead of V0's full-range 89 intervals; therefore this is a **strategy comparison**, not a one-variable causal experiment.


In [ ]:
comparison_specs = [
    ('Final Equity (USDT)', 'final_equity', 1.0),
    ('Net Return (%)', 'net_return', 100.0),
    ('Annualized Return (%)', 'annualized_return', 100.0),
    ('Max Drawdown (%)', 'max_drawdown', 100.0),
    ('Calmar Ratio', 'calmar_ratio', 1.0),
    ('Completed Cycles', 'completed_cycles', 1.0),
    ('Open Positions', 'open_positions', 1.0),
    ('Realized Profit (USDT)', 'realized_profit', 1.0),
    ('Unrealized P&L (USDT)', 'unrealized_pnl', 1.0),
    ('Total Fees (USDT)', 'total_fee_usdt_equiv', 1.0),
    ('Final Cash (USDT)', 'final_cash', 1.0),
    ('Final BTC', 'final_btc', 1.0),
]

comparison_rows = []
for label, key, scale in comparison_specs:
    v0_value = v0_summary[key] * scale
    v1_value = v1_summary[key] * scale
    comparison_rows.append({
        'Metric': label,
        'V0 Fixed Grid': v0_value,
        'V1 Dynamic Grid': v1_value,
        'Difference (V1-V0)': v1_value - v0_value,
    })

comparison_rows.append({
    'Metric': 'Recenter Count',
    'V0 Fixed Grid': 0,
    'V1 Dynamic Grid': v1_summary['recenter_count'],
    'Difference (V1-V0)': v1_summary['recenter_count'],
})

df_comparison = pd.DataFrame(comparison_rows)

print('===== V0 vs V1 COMPARISON =====')
display(df_comparison.round(6))

def monthly_strategy_result(result, initial_capital, prefix):
    eq = result['equity_curve'][['open_time', 'equity']].copy()
    eq['month'] = pd.to_datetime(eq['open_time'], utc=True).dt.strftime('%Y-%m')

    monthly = (
        eq.groupby('month', as_index=False)['equity']
        .last()
        .rename(columns={'equity': f'{prefix}_ending_equity'})
    )

    beginning = monthly[f'{prefix}_ending_equity'].shift(1)
    if len(monthly):
        beginning.iloc[0] = initial_capital

    monthly[f'{prefix}_net_pnl'] = (
        monthly[f'{prefix}_ending_equity'] - beginning
    )
    return monthly

v0_monthly_compare = monthly_strategy_result(v0, BACKTEST_CAPITAL, 'v0')
v1_monthly_compare = monthly_strategy_result(v1, BACKTEST_CAPITAL, 'v1')

df_monthly_comparison = v0_monthly_compare.merge(
    v1_monthly_compare,
    on='month',
)

df_monthly_comparison['equity_difference'] = (
    df_monthly_comparison['v1_ending_equity']
    - df_monthly_comparison['v0_ending_equity']
)
df_monthly_comparison['net_pnl_difference'] = (
    df_monthly_comparison['v1_net_pnl']
    - df_monthly_comparison['v0_net_pnl']
)

print('===== MONTHLY V0 vs V1 =====')
display(df_monthly_comparison.round(2))

plot_df = df_monthly_comparison.copy()
plot_df['month_date'] = pd.to_datetime(plot_df['month'])

plt.figure(figsize=(12, 5))
plt.plot(
    plot_df['month_date'],
    plot_df['v0_ending_equity'],
    label='V0 Fixed Grid',
)
plt.plot(
    plot_df['month_date'],
    plot_df['v1_ending_equity'],
    label='V1 Dynamic Grid',
)
plt.title('BTC Spot Grid — V0 vs V1 Month-End Equity')
plt.xlabel('Month')
plt.ylabel('Equity (USDT)')
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 13. V1 Recenter Diagnostics

In [ ]:
print('===== V1 RECENTER DIAGNOSTICS =====')
print(f"Total re-centers: {len(v1['recenter_log']):,}")

if len(v1['recenter_log']):
    print('First 10 re-centers')
    display(v1['recenter_log'].head(10))
    print('Last 10 re-centers')
    display(v1['recenter_log'].tail(10))

print('Latest regime history')
display(v1['regime_history'].tail(10))


## 14. System Audit

This audit reconciles the historical V0 and V1 results. A backtest result is accepted only if the accounting identities and dynamic re-center rule pass.


In [ ]:
def audit_strategy(result, initial_capital, name):
    summary = result['summary']
    trade_log = result['trade_log']
    equity = result['equity_curve']

    cash_movement_sum = (
        trade_log['cash_movement'].sum() if len(trade_log) else 0.0
    )
    grid_cashflow_sum = (
        trade_log['grid_cashflow'].sum() if len(trade_log) else 0.0
    )

    cash_error = abs(
        initial_capital + cash_movement_sum - summary['final_cash']
    )
    realized_error = abs(
        grid_cashflow_sum - summary['realized_profit']
    )
    equity_error = float(np.max(np.abs(
        equity['cash']
        + equity['btc'] * equity['close']
        - equity['equity']
    )))
    minimum_cash = float(equity['cash'].min())

    return [
        {
            'Strategy': name,
            'Test': 'Cash reconciliation',
            'Expected': '<= 1e-8',
            'Actual': cash_error,
            'Status': 'PASS' if cash_error <= 1e-8 else 'FAIL',
        },
        {
            'Strategy': name,
            'Test': 'Grid Cashflow = realized profit',
            'Expected': '<= 1e-8',
            'Actual': realized_error,
            'Status': 'PASS' if realized_error <= 1e-8 else 'FAIL',
        },
        {
            'Strategy': name,
            'Test': 'Cash + BTC x Close = Equity',
            'Expected': '<= 1e-8',
            'Actual': equity_error,
            'Status': 'PASS' if equity_error <= 1e-8 else 'FAIL',
        },
        {
            'Strategy': name,
            'Test': 'Cash never negative',
            'Expected': '>= 0',
            'Actual': minimum_cash,
            'Status': 'PASS' if minimum_cash >= -1e-8 else 'FAIL',
        },
    ]

audit_rows = []
audit_rows.extend(audit_strategy(v0, BACKTEST_CAPITAL, 'V0 Fixed Grid'))
audit_rows.extend(audit_strategy(v1, BACKTEST_CAPITAL, 'V1 Dynamic Grid'))

if len(v1['recenter_log']):
    recenter_distance_ok = (
        (
            v1['recenter_log']['close']
            - v1['recenter_log']['old_reference']
        ).abs()
        + 1e-9
        >= V1_RECENTER_TRIGGER_GRIDS * V1_GRID_GAP
    ).all()
else:
    recenter_distance_ok = True

audit_rows.append({
    'Strategy': 'V1 Dynamic Grid',
    'Test': 'Recenter threshold respected',
    'Expected': 'True',
    'Actual': bool(recenter_distance_ok),
    'Status': 'PASS' if recenter_distance_ok else 'FAIL',
})

# Monthly P&L reconciliation for both strategies.
for label, result in [('V0 Fixed Grid', v0), ('V1 Dynamic Grid', v1)]:
    monthly = build_monthly_portfolio_pnl(
        result['equity_curve'],
        BACKTEST_CAPITAL,
        START_DATE,
        END_DATE,
    )
    portfolio_gain = (
        result['summary']['final_equity'] - BACKTEST_CAPITAL
    )
    monthly_error = abs(
        monthly['portfolio_net_pnl'].sum() - portfolio_gain
    )
    audit_rows.append({
        'Strategy': label,
        'Test': 'Monthly Portfolio Net P&L = total portfolio gain',
        'Expected': '<= 1e-8',
        'Actual': monthly_error,
        'Status': 'PASS' if monthly_error <= 1e-8 else 'FAIL',
    })

df_system_test_log = pd.DataFrame(audit_rows)
display(df_system_test_log)

failed_system_tests = df_system_test_log.loc[
    df_system_test_log['Status'].ne('PASS')
]

SYSTEM_AUDIT_STATUS = (
    'PASS' if failed_system_tests.empty else 'FAIL'
)
print(f'SYSTEM AUDIT: {SYSTEM_AUDIT_STATUS}')

if SYSTEM_AUDIT_STATUS != 'PASS':
    raise AssertionError('SYSTEM AUDIT FAILED')


## 15. Results

In [ ]:
print('===== FINAL STRATEGY COMPARISON =====')
display(df_comparison.round(6))

print()
print('V0 Fixed Grid')
print(f"  Final Equity : {v0_summary['final_equity']:,.2f} USDT")
print(f"  Net Return   : {v0_summary['net_return']:.2%}")
print(f"  Max Drawdown : {v0_summary['max_drawdown']:.2%}")
print(f"  Calmar       : {v0_summary['calmar_ratio']:.3f}")

print()
print('V1 Dynamic Grid')
print(f"  Final Equity : {v1_summary['final_equity']:,.2f} USDT")
print(f"  Net Return   : {v1_summary['net_return']:.2%}")
print(f"  Max Drawdown : {v1_summary['max_drawdown']:.2%}")
print(f"  Calmar       : {v1_summary['calmar_ratio']:.3f}")
print(f"  Re-centers   : {v1_summary['recenter_count']:,}")

print()
print(f'System Audit  : {SYSTEM_AUDIT_STATUS}')


## 16. Export Latest Backtest Log to GitHub

The log now contains both V0 and V1 in one file: `logs/latest_backtest_log.json`. The Colab Secret `GITHUB_TOKEN` is used only for upload and is never written into the notebook or log.


In [ ]:
def _json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, pd.Period):
        return str(value)
    if pd.isna(value):
        return None
    return value

def _records(df):
    return [
        {k: _json_safe(v) for k, v in row.items()}
        for row in df.to_dict(orient='records')
    ]

log_payload = {
    'log_schema_version': 2,
    'run_info': {
        'generated_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'repository': 'natdanaiii/Trading',
        'branch': 'main',
        'notebook': 'Grid_trading.ipynb',
        'symbol': SYMBOL,
        'start_date': START_DATE,
        'end_date': END_DATE,
        'timeframe': '1m',
        'data_rows': int(len(df_1m)),
        'data_first_time': df_1m['open_time'].min().isoformat(),
        'data_last_time': df_1m['open_time'].max().isoformat(),
    },
    'common_parameters': {
        'initial_capital': BACKTEST_CAPITAL,
        'buy_fee': BUY_FEE,
        'sell_fee': SELL_FEE,
    },
    'v0_parameters': {
        'floor': BACKTEST_FLOOR,
        'ceiling': BACKTEST_CEILING,
        'gap': BACKTEST_GAP,
        'number_of_grids': NUMBER_OF_GRIDS,
        'capital_per_grid': CAPITAL_PER_LEVEL,
        'price_rounding': PRICE_ROUNDING,
        'historical_low': historical_low,
        'historical_high': historical_high,
        'look_ahead_boundary_selection': True,
    },
    'v1_parameters': {
        'gap': V1_GRID_GAP,
        'number_of_grids': V1_NUMBER_OF_GRIDS,
        'capital_per_grid': V1_CAPITAL_PER_GRID,
        'recenter_trigger_grids': V1_RECENTER_TRIGGER_GRIDS,
        'recenter_trigger_usdt': (
            V1_RECENTER_TRIGGER_GRIDS * V1_GRID_GAP
        ),
        'compounding': False,
        'causal_recenter': True,
    },
    'verification': {
        'v0_status': (
            'PASS' if verification_failed.empty else 'FAIL'
        ),
        'v0_scenarios': _records(df_verification_report),
        'v1_status': (
            'PASS' if v1_verification_failed.empty else 'FAIL'
        ),
        'v1_scenarios': _records(df_v1_verification),
    },
    'system_audit': {
        'status': SYSTEM_AUDIT_STATUS,
        'checks': _records(df_system_test_log),
    },
    'v0_summary': {
        k: _json_safe(v) for k, v in v0_summary.items()
    },
    'v1_summary': {
        k: _json_safe(v) for k, v in v1_summary.items()
    },
    'comparison': _records(df_comparison),
    'monthly_comparison': _records(df_monthly_comparison),
    'v0_monthly_grid_cashflow': _records(
        df_v0_grid_cashflow_monthly
    ),
    'recenter_diagnostics': {
        'count': int(len(v1['recenter_log'])),
        'first_10': _records(v1['recenter_log'].head(10)),
        'last_10': _records(v1['recenter_log'].tail(10)),
    },
}

LOCAL_LOG_PATH = '/content/latest_backtest_log.json'
with open(LOCAL_LOG_PATH, 'w') as f:
    json.dump(log_payload, f, indent=2, allow_nan=False)

print('===== BACKTEST LOG =====')
print(f'Local log    : {LOCAL_LOG_PATH}')
print(
    'Verification : '
    f"V0 {'PASS' if verification_failed.empty else 'FAIL'} / "
    f"V1 {'PASS' if v1_verification_failed.empty else 'FAIL'}"
)
print(f'System Audit : {SYSTEM_AUDIT_STATUS}')

try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None

if not github_token:
    print()
    print("GitHub upload SKIPPED: Colab Secret 'GITHUB_TOKEN' was not found.")
else:
    repo = 'natdanaiii/Trading'
    path = 'logs/latest_backtest_log.json'
    api_url = f'https://api.github.com/repos/{repo}/contents/{path}'

    headers = {
        'Authorization': f'Bearer {github_token}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
    }

    import requests

    existing = requests.get(api_url, headers=headers, timeout=30)
    existing_sha = (
        existing.json().get('sha')
        if existing.status_code == 200
        else None
    )

    encoded = base64.b64encode(
        json.dumps(log_payload, indent=2).encode()
    ).decode()

    body = {
        'message': 'Update latest V0 V1 backtest log',
        'content': encoded,
        'branch': 'main',
    }
    if existing_sha:
        body['sha'] = existing_sha

    upload = requests.put(
        api_url,
        headers=headers,
        json=body,
        timeout=30,
    )
    upload.raise_for_status()

    print()
    print('GitHub upload : SUCCESS')
    print(f'Path          : {path}')
    print(f"Commit SHA    : {upload.json()['commit']['sha']}")
